# Model Training — FurnishAI / Ikarus
Trains: NLP (TF-IDF + KMeans), Embedding validation, CV (ResNet18 fine-tune).
Saves weights to `backend/models/` for use by the FastAPI services.


In [ ]:
import sys; sys.path.insert(0, '../backend')
import pandas as pd, numpy as np, ast, re, os, warnings, joblib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Load & clean ikarus dataset ──────────────────────────────────────
from utils.data_utils import load_and_clean
df = load_and_clean('../backend/data/intern_data_ikarus.csv')
print(f'Loaded: {len(df)} products')
print(df[['title','price','leaf_category']].head(4))

In [ ]:
# ── NLP: TF-IDF vectorisation ────────────────────────────────────────
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

nlp_spacy = spacy.load('en_core_web_sm')

def preprocess(text):
    text = re.sub(r'[^a-z0-9\s]',' ', str(text).lower())
    text = re.sub(r'\s+',' ',text).strip()
    doc  = nlp_spacy(text)
    return ' '.join(t.lemma_ for t in doc if not t.is_stop and t.is_alpha)

corpus = (
    df['title'] + ' ' + df['description'] + ' ' +
    df['leaf_category'] + ' ' + df['leaf_category']
).apply(preprocess)

vectorizer   = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'TF-IDF shape: {tfidf_matrix.shape}')

In [ ]:
# ── Find optimal K via silhouette score ──────────────────────────────
svd     = TruncatedSVD(n_components=50, random_state=42)
reduced = svd.fit_transform(tfidf_matrix)

scores, ks = [], range(4,16,2)
for k in ks:
    km  = KMeans(n_clusters=k, random_state=42, n_init=5)
    lbl = km.fit_predict(reduced)
    s   = silhouette_score(reduced, lbl)
    scores.append(s)
    print(f'K={k:2d}: silhouette={s:.4f}')

best_k = list(ks)[np.argmax(scores)]
print(f'Best K = {best_k}')

In [ ]:
# ── Train final KMeans ────────────────────────────────────────────────
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(reduced)
for c in range(best_k):
    titles = df[df['cluster']==c]['title'].head(3).tolist()
    print(f'Cluster {c}: {[t[:40] for t in titles]}')

In [ ]:
# ── Validate SentenceTransformer embeddings (same as Pinecone) ────────
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = (df['title'] + ' ' + df['leaf_category']).tolist()
print('Encoding...')
embs  = model.encode(texts, show_progress_bar=True)

queries = ['comfortable ottoman living room','bar stool kitchen','wall mirror hallway']
for q in queries:
    sims = cosine_similarity(model.encode([q]), embs)[0]
    top3 = sims.argsort()[::-1][:3]
    print(f'\nQuery: "{q}"')
    for i in top3:
        print(f'  [{sims[i]:.3f}] {df.iloc[i]["title"][:70]}')

In [ ]:
# ── CV: Map leaf categories to label indices ──────────────────────────
CAT_LIST = ['Doormats','End Tables','Ottomans','Wall-Mounted Mirrors',
            'Barstools','Chairs','Towel Bars','Home Office Desk Chairs',
            'Home Office Desks','Bean Bags','Sofas','Storage','Other']

def to_label(cat):
    cat = str(cat).strip()
    return CAT_LIST.index(cat) if cat in CAT_LIST else CAT_LIST.index('Other')

df['label'] = df['leaf_category'].apply(to_label)
print('Label distribution:'); print(df['label'].value_counts().sort_index())

In [ ]:
# ── CV: PyTorch Dataset streaming Amazon images ───────────────────────
import torch, torch.nn as nn, requests
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
from io import BytesIO

class IkarusDS(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label'])
        url   = str(row.get('first_image','')).strip()
        try:
            r   = requests.get(url, timeout=4, headers={'User-Agent':'Mozilla/5.0'})
            img = Image.open(BytesIO(r.content)).convert('RGB')
        except:
            img = Image.new('RGB',(224,224),(210,210,210))
        if self.transform: img = self.transform(img)
        return img, label

tfm_train = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tfm_val = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

img_df = df[df['first_image'].notna() & (df['first_image']!='')].reset_index(drop=True)
split  = int(len(img_df)*0.8)
train_dl = DataLoader(IkarusDS(img_df[:split], tfm_train), batch_size=16, shuffle=True, num_workers=0)
val_dl   = DataLoader(IkarusDS(img_df[split:], tfm_val),   batch_size=16, shuffle=False,num_workers=0)
print(f'Train: {len(img_df[:split])}  Val: {len(img_df[split:])}')

In [ ]:
# ── Train ResNet18 (frozen backbone, fine-tune layer4 + FC) ──────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

cv_model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for name, p in cv_model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        p.requires_grad = False
cv_model.fc = nn.Linear(cv_model.fc.in_features, len(CAT_LIST))
cv_model = cv_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, cv_model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

for epoch in range(5):
    cv_model.train(); tloss = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(cv_model(imgs), labels)
        loss.backward(); optimizer.step()
        tloss += loss.item()
    cv_model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            preds   = cv_model(imgs).argmax(1)
            correct += (preds==labels).sum().item()
            total   += len(labels)
    acc = correct/total if total else 0
    print(f'Epoch {epoch+1}/5  loss={tloss/len(train_dl):.4f}  val_acc={acc:.4f}')
    scheduler.step()
print('Done.')

In [ ]:
# ── Save all models ───────────────────────────────────────────────────
os.makedirs('../backend/models', exist_ok=True)
torch.save(cv_model.state_dict(), '../backend/models/image_classifier.pth')
joblib.dump(vectorizer, '../backend/models/tfidf_vectorizer.pkl')
joblib.dump(kmeans,     '../backend/models/kmeans_model.pkl')
print('Saved to ../backend/models/')